# 04 · Models and evaluation
**Owner:** Kasula Lalithendra

1. **train**: tune Logistic Regression, Random Forest and Gradient Boosting with GroupKFold (by premise) and class weights
2. **validation**: pick the best model and its decision threshold
3. **test**: evaluate once: precision, recall, F1, confusion matrix, confidence intervals and error analysis

In [1]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, f1_score, precision_recall_curve, precision_score, recall_score
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import Pipeline

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
PROCESSED = ROOT / "data" / "processed"
warnings.filterwarnings("ignore")

from src.features import add_structured_features, make_preprocessor

SEED = 42
train, validation, test = (add_structured_features(pd.read_csv(PROCESSED / f"{s}.csv"))
                           for s in ["train", "validation", "test"])


def scores(y, pred):
    return {"precision": precision_score(y, pred, zero_division=0), "recall": recall_score(y, pred, zero_division=0),
            "f1": f1_score(y, pred, zero_division=0)}

## 1. Tune on train
GroupKFold keeps all rows of a premise in the same fold. Class weights make the rare valid class count as much as the
invalid one. The full pipeline (TF-IDF included) is re-fitted in every fold.

In [2]:
models = {
    "Logistic Regression": (LogisticRegression(max_iter=3000, class_weight="balanced"),
                            {"model__C": [0.1, 1, 10]}, False),
    "Random Forest": (RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=SEED, n_jobs=-1),
                      {"model__max_depth": [None, 20], "model__min_samples_leaf": [1, 5]}, False),
    "Gradient Boosting": (HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED),
                          {"model__learning_rate": [0.05, 0.1], "model__max_leaf_nodes": [15, 31]}, True),
}

searches = {}
for name, (model, grid, dense) in models.items():
    pipe = Pipeline([("features", make_preprocessor("both", dense=dense)), ("model", model)])
    searches[name] = GridSearchCV(pipe, grid, cv=GroupKFold(5), scoring="f1", n_jobs=-1)
    searches[name].fit(train, train["label"], groups=train["premise_id"])
    print(f"{name:<20} best CV F1 = {searches[name].best_score_:.3f}  {searches[name].best_params_}")

Logistic Regression  best CV F1 = 0.829  {'model__C': 10}


Random Forest        best CV F1 = 0.911  {'model__max_depth': None, 'model__min_samples_leaf': 1}


Gradient Boosting    best CV F1 = 0.918  {'model__learning_rate': 0.1, 'model__max_leaf_nodes': 31}


## 2. Pick the model on validation

In [3]:
val_proba = {name: s.predict_proba(validation)[:, 1] for name, s in searches.items()}
val_table = pd.DataFrame({name: scores(validation["label"], p >= 0.5) for name, p in val_proba.items()}).T.round(3)
val_table

,precision,recall,f1
Logistic Regression,0.714,0.891,0.793
Random Forest,0.828,0.969,0.893
Gradient Boosting,0.810,0.860,0.835


In [4]:
best = val_table["f1"].idxmax()
precision, recall, thresholds = precision_recall_curve(validation["label"], val_proba[best])
f1 = 2 * precision * recall / np.clip(precision + recall, 1e-9, None)
threshold = thresholds[np.argmax(f1[:-1])]
print(f"chosen model: {best}, threshold: {threshold:.3f}")

chosen model: Random Forest, threshold: 0.607


We take the model with the best validation F1 and the threshold that maximises F1 on validation. Nothing changes after this point.